## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cassert>
using namespace std;

inline int fastRead() {
    int res = 0;
    char c = getchar();
    while (c < '0' || c > '9') c = getchar();
    while (c >= '0' && c <= '9') {
        res = res * 10 + (c - '0');
        c = getchar();
    }
    return res;
}

const int MAX_SIZE = 1030;

int N;
int valA, valB;
int LsbLen;
int arr[MAX_SIZE], loc[MAX_SIZE];
vector<int> opsList;

void updateLocations() {
    for (int i = 0; i < N; ++i) {
        loc[arr[i]] = i;
    }
}

void opSwap() {
    opsList.push_back(0);
    for (int i = 0; i < N; ++i) {
        if (arr[i] == valA) {
            arr[i] = valB;
        } else if (arr[i] == valB) {
            arr[i] = valA;
        }
    }
    updateLocations();
}

void opShift(int delta) {
    delta = (delta % N + N) % N;
    if (delta == 0) return;
    opsList.push_back(delta);
    for (int i = 0; i < N; ++i) {
        arr[i] = (arr[i] + delta) % N;
    }
    updateLocations();
}

void opXor(int mask) {
    if (mask == 0) return;
    opsList.push_back(-mask);
    for (int i = 0; i < N; ++i) {
        arr[i] ^= mask;
    }
    updateLocations();
}

// 计算坐标对
void findCoordinates(int u, int v, int &posU, int &posV) {
    int diff = (v - u + N - LsbLen + N) % N;
    posU = 0; 
    posV = 0;
    
    for (int step = N / 2; step >= 2 * LsbLen; step >>= 1) {
        if (diff >= step) {
            diff -= step;
            posV += step / 2;
        } else {
            posU += step / 2;
        }
    }
    posU += (N / 2);
    posU += (u & (LsbLen - 1));
    posV += (u & (LsbLen - 1));
}

// 核心交换逻辑
void executeSwap(int u, int v) {
    if ((u / LsbLen) % 2 == (v / LsbLen) % 2) {
        int pivot = ((u / LsbLen) % 2 == 0) ? ((u & (LsbLen - 1)) + LsbLen) : (u & (LsbLen - 1));
        executeSwap(u, pivot);
        executeSwap(v, pivot);
        executeSwap(u, pivot);
        return;
    }
    int pA, pB, pU, pV;
    findCoordinates(valA, valB, pA, pB);
    findCoordinates(u, v, pU, pV);
    opShift((pU - u + N) % N);
    opXor(pU ^ pA);
    opShift((valA - pA + N) % N);
    opSwap();
    opShift((pA - valA + N) % N);
    opXor(pU ^ pA);
    opShift((u - pU + N) % N);
}

// 置换状态结构体
struct PermState {
    int seq[MAX_SIZE];
    int sz;
    vector<int> moves;
    bool operator<(const PermState &other) const {
        for (int i = 0; i < sz; ++i) {
            if (seq[i] != other.seq[i]) {
                return seq[i] < other.seq[i];
            }
        }
        return false;
    }
    bool isSorted() const {
        for (int i = 0; i < sz - 1; ++i) {
            if (seq[i] > seq[i + 1]) return false;
        }
        return true;
    }
    PermState getInverse() const {
        PermState res;
        res.sz = sz;
        for (int i = 0; i < sz; ++i) {
            res.seq[seq[i]] = i;
        }
        return res;
    }
    bool construct() {
        vector<bool> seen(100005, false);
        for (int i = 0; i < sz; ++i) seen[i] = true;
        for (int i = 0; i < sz; ++i) {
            if (!seen[i]) return false;
        }
        if (sz == 1) return true;
        PermState leftHalf, rightHalf;
        leftHalf.sz = rightHalf.sz = sz / 2;
        for (int i = 0; i < sz / 2; ++i) {
            leftHalf.seq[i] = seq[i * 2] / 2;
            rightHalf.seq[i] = seq[i * 2 + 1] / 2;
        }
        if (!leftHalf.construct() || !rightHalf.construct()) {
            return false;
        }
        if (seq[0] & 1) {
            moves.push_back(sz == 2 ? 1 : -1);
        }
        int xorLeft = 0;
        for (int elem : leftHalf.moves) {
            if (elem > 0) {
                moves.push_back(-1);
                moves.push_back(1);
            } else {
                moves.push_back(elem * 2);
                xorLeft ^= (-elem) * 2;
            }
        }
        if (xorLeft) moves.push_back(-xorLeft);
        int xorRight = 0;
        for (int elem : rightHalf.moves) {
            if (elem > 0) {
                moves.push_back(1);
                moves.push_back(-1);
            } else {
                moves.push_back(elem * 2);
                xorRight ^= (-elem) * 2;
            }
        }
        if ((xorRight & (sz / 2)) != (xorLeft & (sz / 2))) {
            return false;
        }
        if (xorLeft >= sz / 2) xorLeft -= sz / 2;
        if (xorRight >= sz / 2) xorRight -= sz / 2;
        if (xorLeft != xorRight) return false;
        vector<int> optimizedMoves;
        for (int m : moves) {
            if (optimizedMoves.empty()) {
                optimizedMoves.push_back(m);
            } else {
                if (m < 0 && optimizedMoves.back() < 0) {
                    optimizedMoves.back() = -((-optimizedMoves.back()) ^ (-m));
                    if (optimizedMoves.back() == 0) {
                        optimizedMoves.pop_back();
                    }
                } else {
                    optimizedMoves.push_back(m);
                }
            }
        }
        moves.swap(optimizedMoves);
        return true;
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    N = fastRead();
    valA = fastRead();
    valB = fastRead();
    for (int i = 0; i < N; ++i) {
        arr[i] = fastRead();
    }
    updateLocations();
    LsbLen = (valA - valB + N) % N;
    LsbLen &= -LsbLen;
    if (LsbLen == 0) {
        LsbLen = N;
    }
    if (LsbLen > 1) {
        PermState baseState;
        baseState.sz = LsbLen;
        for (int i = 0; i < N; ++i) {
            baseState.seq[i] = arr[i] & (LsbLen - 1);
        }
        if (!baseState.construct()) {
            printf("-1\n");
            return 0;
        }
        for (int op : baseState.moves) {
            if (op > 0) {
                opShift(op);
            } else {
                opXor(-op);
            }
        }
    }
    for (int rem = 0; rem < LsbLen; ++rem) {
        vector<int> group;
        for (int j = rem; j < N; j += LsbLen) {
            group.push_back(arr[j]);
        }
        sort(group.begin(), group.end());
        bool isValid = true;
        int index = 0;
        for (int j = rem; j < N; j += LsbLen) {
            if (group[index++] != j) {
                isValid = false;
                break;
            }
        }
        if (!isValid) {
            printf("-1\n");
            return 0;
        }
        for (int j = rem; j < N; j += LsbLen) {
            if (arr[j] != j) {
                executeSwap(j, arr[j]);
            }
        }
    }
    for (int i = 0; i < N; ++i) {
        assert(arr[i] == i);
    }
    printf("%d\n", (int)opsList.size());
    for (int step : opsList) {
        if (step == 0) {
            printf("0\n");
        } else if (step < 0) {
            printf("1 %d\n", -step);
        } else {
            printf("2 %d\n", step);
        }
    }
    return 0;
}

## B 长跑

In [ ]:
import sys
from collections import deque

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    ptr = 0
    n_len = len(input_data)
    out = []
    while ptr < n_len:
        try:
            N = int(input_data[ptr])
            L = int(input_data[ptr+1])
            Maxn = int(input_data[ptr+2])
            S = int(input_data[ptr+3])
            ptr += 4
        except IndexError:
            break
            
        stations = []
        for _ in range(N):
            p = int(input_data[ptr])
            c = int(input_data[ptr+1])
            if p < L:
                stations.append((p, c))
            ptr += 2
        stations.sort(key=lambda x: x[0])
        nodes = [(0, 0)] + stations + [(L, 0)]
        M = len(nodes)
        dp = [float('inf')] * M
        V = [float('inf')] * M
        dp[0] = 0
        V[0] = 0
        q = deque([0])
        possible = True
        
        for i in range(1, M):
            pos_i, cost_i = nodes[i]
            while q and pos_i - nodes[q[0]][0] > Maxn:
                q.popleft()
            if not q:
                if i == M - 1: 
                    possible = False
                dp[i] = float('inf')
                V[i] = float('inf')
                continue
            dp[i] = V[q[0]]
            V[i] = dp[i] + cost_i
            while q and V[q[-1]] >= V[i]:
                q.pop()
            q.append(i)
        if possible and dp[-1] <= S:
            out.append("Yes")
        else:
            out.append("No")
    print('\n'.join(out))

if __name__ == '__main__':
    solve()

## C 最长回文

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    ptr = 0
    n_len = len(input_data)
    out = []

    def manacher(s):
        T = "^#" + "#".join(s) + "#$"
        P = [0] * len(T)
        C = R = 0
        for i in range(1, len(T) - 1):
            if R > i:
                i_mirror = 2 * C - i
                P[i] = min(R - i, P[i_mirror])
            while T[i + 1 + P[i]] == T[i - 1 - P[i]]:
                P[i] += 1
                
            if i + P[i] > R:
                C, R = i, i + P[i]
        return P

    while ptr < n_len:
        n = int(input_data[ptr])
        A = input_data[ptr+1]
        B = input_data[ptr+2]
        ptr += 3
        PA = manacher(A)
        PB = manacher(B)
        revA = A[::-1] 
        max_len = 0
        for idx in range(1, len(PA) - 1):
            radius = PA[idx]
            start_A = (idx - radius - 1) // 2
            end_A = (idx + radius - 1) // 2 - 1
            k = end_A
            if k < 0: 
                if radius > max_len: max_len = radius
                continue

            pos_revA = n - start_A
            pos_B = k
            max_L = start_A if start_A < n - pos_B else n - pos_B
            if max_L < 0: max_L = 0
            req_L = (max_len - radius) // 2 + 1
            if req_L < 0: req_L = 0
            if req_L > max_L:
                continue   
            if req_L > 0:
                if revA[pos_revA : pos_revA + req_L] != B[pos_B : pos_B + req_L]:
                    continue
            low = req_L
            high = max_L
            best = req_L
            while low <= high:
                mid = (low + high) // 2
                if revA[pos_revA : pos_revA + mid] == B[pos_B : pos_B + mid]:
                    best = mid
                    low = mid + 1
                else:
                    high = mid - 1
            cand = radius + 2 * best
            if cand > max_len:
                max_len = cand

        for idx in range(1, len(PB) - 1):
            radius = PB[idx]
            if radius == 0: 
                continue   
            start_B = (idx - radius - 1) // 2
            end_B = (idx + radius - 1) // 2 - 1
            k = start_B
            if k >= n:
                if radius > max_len: max_len = radius
                continue
                
            pos_revA = n - k - 1
            pos_B = end_B + 1
            
            max_L = k + 1 if k + 1 < n - pos_B else n - pos_B
            if max_L < 0: max_L = 0
            req_L = (max_len - radius) // 2 + 1
            if req_L < 0: req_L = 0 
            if req_L > max_L:
                continue  
            if req_L > 0:
                if revA[pos_revA : pos_revA + req_L] != B[pos_B : pos_B + req_L]:
                    continue
                    
            low = req_L
            high = max_L
            best = req_L
            while low <= high:
                mid = (low + high) // 2
                if revA[pos_revA : pos_revA + mid] == B[pos_B : pos_B + mid]:
                    best = mid
                    low = mid + 1
                else:
                    high = mid - 1
            cand = radius + 2 * best
            if cand > max_len:
                max_len = cand
        out.append(str(max_len))
    print('\n'.join(out))
    
if __name__ == '__main__':
    solve()

## D 优惠券

In [ ]:
import sys
from bisect import bisect_right

def solve():
    def line_generator():
        for line in sys.stdin:
            line = line.strip()
            if line:
                yield line
                
    gen = line_generator()
    out = []
    
    while True:
        try:
            m_str = next(gen)
        except StopIteration:
            break
        try:
            m = int(m_str)
        except ValueError:
            continue
        error_line = -1
        owned = {}     
        last_I = {}     
        last_O = {}     
        q_list = []     
        dsu = [0]       

        def find(i):
            path = []
            while dsu[i] != i:
                path.append(i)
                i = dsu[i]
            for node in path:
                dsu[node] = i
            return i

        for i in range(1, m + 1):
            try:
                record = next(gen)
            except StopIteration:
                break

            if error_line != -1:
                continue
            
            parts = record.split()
            if not parts:
                continue
            
            op = parts[0]
            if op == '?':
                q_list.append(i)
                dsu.append(len(q_list)) 
            elif op == 'I' or op == 'O':
                if len(parts) < 2:
                    continue
                try:
                    val = int(parts[1])
                except ValueError:
                    continue
                    
                if op == 'I':
                    is_owned = owned.get(val, False)
                    if is_owned:
                        req_last = last_I.get(val, 0)
                        start_idx = bisect_right(q_list, req_last)
                        true_idx = find(start_idx)
                        if true_idx < len(q_list):
                            dsu[true_idx] = find(true_idx + 1)
                            last_I[val] = i 
                        else:
                            error_line = i
                    else:
                        owned[val] = True
                        last_I[val] = i
                        
                elif op == 'O':
                    is_owned = owned.get(val, False)
                    if not is_owned:
                        req_last = last_O.get(val, 0)
                        start_idx = bisect_right(q_list, req_last)
                        true_idx = find(start_idx)
                        
                        if true_idx < len(q_list):
                            dsu[true_idx] = find(true_idx + 1)
                            last_O[val] = i
                        else:
                            error_line = i
                    else:
                        owned[val] = False
                        last_O[val] = i

        out.append(str(error_line))
    if out:
        print('\n'.join(out))

if __name__ == '__main__':
    sys.setrecursionlimit(200000)
    solve()

## E 任意点

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    points = []
    idx = 1
    for _ in range(n):
        x = int(input_data[idx])
        y = int(input_data[idx+1])
        points.append((x, y))
        idx += 2
    visited = [False] * n
    
    def dfs(u):
        visited[u] = True
        for v in range(n):
            if not visited[v]:
                if points[u][0] == points[v][0] or points[u][1] == points[v][1]:
                    dfs(v)
    components = 0
    
    for i in range(n):
        if not visited[i]:
            components += 1
            dfs(i)
    print(components - 1)

if __name__ == '__main__':
    sys.setrecursionlimit(2000)
    solve()

## F 通配符匹配

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    pattern = input_data[0]
    n = int(input_data[1])
    filenames = input_data[2:]

    class ExactMatcher:
        def __init__(self, p):
            self.L = len(p)
            self.parts = []
            curr = 0
            for part in p.split('?'):
                if part:
                    self.parts.append((part, curr))
                curr += len(part) + 1

        def match(self, S, start_idx):
            if start_idx < 0 or start_idx + self.L > len(S):
                return False
            for p, off in self.parts:
                if S[start_idx + off : start_idx + off + len(p)] != p:
                    return False
            return True

    class MidMatcher:
        def __init__(self, p):
            self.L = len(p)
            self.parts = []
            curr = 0
            for part in p.split('?'):
                if part:
                    self.parts.append((part, curr))
                curr += len(part) + 1
                
        def match_first(self, S, start, end):
            if start + self.L > end:
                return -1
                
            if not self.parts:
                return start
                
            best_part = None
            best_off = -1
            min_count = float('inf')
            for p, off in self.parts:
                c = S.count(p, start, end)
                if c < min_count:
                    min_count = c
                    best_part = p
                    best_off = off
                    
            search_idx = start + best_off
            limit = end - self.L + best_off + len(best_part)
            
            while True:
                idx = S.find(best_part, search_idx, limit)
                if idx == -1:
                    return -1
                    
                base_pos = idx - best_off
                match_ok = True
                for p, off in self.parts:
                    if off == best_off: 
                        continue
                    if S[base_pos + off : base_pos + off + len(p)] != p:
                        match_ok = False
                        break
                        
                if match_ok:
                    return base_pos
                search_idx = idx + 1

    blocks = pattern.split('*')
    out = []

    if len(blocks) == 1:
        matcher = ExactMatcher(blocks[0])
        for s in filenames:
            if len(s) == matcher.L and matcher.match(s, 0):
                out.append("YES")
            else:
                out.append("NO")
        print('\n'.join(out))
        return
    pref = blocks[0]
    suff = blocks[-1]
    mids = blocks[1:-1]
    min_required_len = sum(len(b) for b in blocks)
    pref_matcher = ExactMatcher(pref) if pref else None
    suff_matcher = ExactMatcher(suff) if suff else None
    mid_matchers = [MidMatcher(b) for b in mids if b]
    
    for s in filenames:
        if len(s) < min_required_len:
            out.append("NO")
            continue
            
        if pref_matcher and not pref_matcher.match(s, 0):
            out.append("NO")
            continue
            
        if suff_matcher and not suff_matcher.match(s, len(s) - len(suff)):
            out.append("NO")
            continue

        curr_start = len(pref)
        end_limit = len(s) - len(suff)
        possible = True
        for matcher in mid_matchers:
            pos = matcher.match_first(s, curr_start, end_limit)
            if pos == -1:
                possible = False
                break
            curr_start = pos + matcher.L
        if possible:
            out.append("YES")
        else:
            out.append("NO")
    print('\n'.join(out))

if __name__ == '__main__':
    solve()

## G 汉诺塔

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    n = int(input_data[0])
    priorities = input_data[1:]
    pegs = ['A', 'B', 'C']

    def get_other(p1, p2):
        for p in pegs:
            if p != p1 and p != p2:
                return p

    target = [{'A': '', 'B': '', 'C': ''} for _ in range(n + 1)]
    cost = [{'A': 0, 'B': 0, 'C': 0} for _ in range(n + 1)]

    for X in pegs:
        dests = [p for p in pegs if p != X]
        op1 = X + dests[0]
        op2 = X + dests[1]
        if priorities.index(op1) < priorities.index(op2):
            target[1][X] = dests[0]
        else:
            target[1][X] = dests[1]
        cost[1][X] = 1

    for i in range(2, n + 1):
        for X in pegs:
            Y = target[i-1][X]
            Z = get_other(X, Y)
            W = target[i-1][Y]
            if W == Z:
                target[i][X] = Z
                cost[i][X] = cost[i-1][X] + 1 + cost[i-1][Y]
            elif W == X:
                target[i][X] = Y
                cost[i][X] = cost[i-1][X] * 2 + cost[i-1][Y] + 2

    print(cost[n]['A'])

if __name__ == '__main__':
    solve()

## H 马步距离

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    xp = int(input_data[0])
    yp = int(input_data[1])
    xs = int(input_data[2])
    ys = int(input_data[3])
    dx = abs(xp - xs)
    dy = abs(yp - ys)

    if dx < dy:
        dx, dy = dy, dx
    if dx == 1 and dy == 0:
        print(3)
        return
    if dx == 2 and dy == 2:
        print(4)
        return
    ans = max((dx + 1) // 2, (dx + dy + 2) // 3)
    if (ans % 2) != ((dx + dy) % 2):
        ans += 1
    print(ans)

if __name__ == '__main__':
    solve()

## I 直方图最大矩形

In [ ]:
from typing import List

class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        new_heights = [0] + heights + [0]
        stack = []  
        max_area = 0
        
        for i in range(len(new_heights)):
            while stack and new_heights[i] < new_heights[stack[-1]]:
                h = new_heights[stack.pop()]
                w = i - stack[-1] - 1
                max_area = max(max_area, h * w)
            stack.append(i)
        return max_area

## J 消防局的设立

In [ ]:
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    if n == 0:
        print(0)
        return
        
    adj = [[] for _ in range(n + 1)]
    for i in range(2, n + 1):
        u = i
        v = int(input_data[i - 1])
        adj[u].append(v)
        adj[v].append(u)
        
    parent = [0] * (n + 1)
    order = []
    q = [1]
    visited = [False] * (n + 1)
    visited[1] = True
    
    for curr in q:
        order.append(curr)
        for neighbor in adj[curr]:
            if not visited[neighbor]:
                visited[neighbor] = True
                parent[neighbor] = curr
                q.append(neighbor)
    order.reverse()
    dist = [float('inf')] * (n + 1)
    ans = 0
    for u in order:
        if dist[u] <= 2:
            continue
        v = u
        if parent[v] != 0:
            v = parent[v]
        if parent[v] != 0:
            v = parent[v]
        ans += 1
        dist[v] = 0
        q2 = [v]
        for step in range(2):
            next_q2 = []
            for curr in q2:
                curr_dist = dist[curr]
                for neighbor in adj[curr]:
                    if dist[neighbor] > curr_dist + 1:
                        dist[neighbor] = curr_dist + 1
                        next_q2.append(neighbor)
            q2 = next_q2
            
    print(ans)

if __name__ == '__main__':
    solve()